In [32]:
# -------- Imports --------
import sys
import os
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from _utility import *

# -- Qiskit imports --
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.synthesis import MatrixExponential
from qiskit.quantum_info import SparsePauliOp, Pauli, Operator
from qiskit.primitives import StatevectorEstimator

np.random.seed(0)

In [33]:
# -------- Parameters --------
# -- Grid parameters --
Nx, Ny, Nz = 16, 16, 16
dx, dy, dz = 0.05, 0.05, 0.05

# Calculate staggered grid sizes for 3D elastic
N_main = Nx * Ny * Nz
N_vx = (Nx-1) * Ny * Nz
N_vy = Nx * (Ny-1) * Nz
N_vz = Nx * Ny * (Nz-1)
N_sxy = (Nx-1) * (Ny-1) * Nz
N_sxz = (Nx-1) * Ny * (Nz-1)
N_syz = Nx * (Ny-1) * (Nz-1)
N_vel = N_vx + N_vy + N_vz
N_stress = 3*N_main + N_sxy + N_sxz + N_syz
psi_len = N_vel + N_stress  # Total state vector size for 3D elastic

# -- Simulation parameters --
t = 5.0

# -- Subspace projector --
#mask = np.random.choice([0, 1], size=psi_len, p=[0.9, 0.1])

#Define a function to get the indices of the points for the subspace energy calculation
def get_mask_indices(ith_element):
    subspace_points = []
    for i in range(len(ith_element)):
        for j in ith_element[i]: 
            if i == 0:
                subspace_points.append(j)
                subspace_points.append(N_vel+3*N_main+N_sxy+j)
            elif i == 1:
                subspace_points.append(N_vx+j)
                subspace_points.append(N_vel+3*N_main+N_sxy+N_sxz+j)
            elif i == 2:
                subspace_points.append(N_vx+N_vy+j)
                subspace_points.append(N_vel+j)
                subspace_points.append(N_vel+N_main+j)
                subspace_points.append(N_vel+2*N_main+j)
            elif i == 3: 
                subspace_points.append(N_vel+3*N_main+j)
    print(f'subspace_points {type(subspace_points)}: {subspace_points}')
    return list(filter(lambda x: x < psi_len, subspace_points))
def get_i_location(Nx,Ny,Nz,z_min,z_max):
    assert z_min >= 0 and z_max <= Nz and z_min < z_max
    i_location = []
    #Should be 4 different unique amounts of grid points for the same z location, the main grid, the points for stress_xy, vx, and vy
    #vx
    i_location.append(np.arange((Nx-1)*Ny*z_min,(Nx-1)*Ny*z_max,1))
    #vy
    i_location.append(np.arange(Nx*(Ny-1)*z_min,Nx*(Ny-1)*z_max,1))
    #main
    i_location.append(np.arange(Nx*Ny*z_min,Nx*Ny*z_max,1))
    #stress_xy
    i_location.append(np.arange((Nx-1)*(Ny-1)*z_min,(Nx-1)*(Ny-1)*z_max,1))

    print(f'i_location {type(i_location)}: {i_location}')
    return i_location
#Define an array of i points, for example, all points with the same first z coordinate range (up to 1/2 step away from the center)
mask = subspaceProjector(psi_len,get_mask_indices(get_i_location(Nx,Ny,Nz,0,1)))
print(f"Total points in subspace: {sum(mask)}")

i_location <class 'list'>: [array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
       104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
       117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
       130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142,
       143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155,
       156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168,
       169, 170, 171, 172, 173, 174,

In [ ]:
# A function returning a dictionary of masks for kinetic, potential, and total energy subspace projectors
def threeProjectors(mask, Nx, Ny, Nz):
    kinetic = np.copy(mask)
    potential = np.copy(mask)
    kinetic_num = (Nx-1)*Ny*Nz + Nx*(Ny-1)*Nz + Nx*Ny*(Nz-1)
    kinetic[kinetic_num:] = 0
    potential[:kinetic_num] = 0
    return {
        'kinetic' : kinetic,
        'potential' : potential,
        'total' : mask
    }

mask_dict = threeProjectors(mask, Nx, Ny, Nz)
print(mask_dict)

{'kinetic': array([1., 1., 1., ..., 0., 0., 0.], shape=(34608,)), 'potential': array([0., 0., 0., ..., 0., 0., 0.], shape=(34608,)), 'total': array([1., 1., 1., ..., 0., 0., 0.], shape=(34608,))}


: 

In [ ]:
# Important: The following implementation of the material parameters is not scalable. This code is only for demonstration purposes and assumes oracle access. 
# In a real-world scenario, the material parameters and the Hamiltonian need to be sparsly constructed on the QC itself.
# Furthermore, this code uses direct matrix exponentiation, which is inefficient for large matrices. Other integration methods should be used (e.g. Trotter-Suzuki, Qubitization).

# -- Material properties (Oracle Access) -- 
# P-wave and S-wave velocities
c_p = 3.0  # P-wave velocity
c_s = 1.5  # S-wave velocity
rho0 = 2.0  # Density

# Density model and S matrix (3D with fractures)
ADD_FRACTURES = True
rho_model, S_matrix = rho_model_compliance_matrix(Nx,Ny,Nz,dx,dy,dz,ADD_FRACTURES)

# -------- Simulation (3D Elastic) (Oracle Access) --------
(H, A, _, B_sqrt, B_inv, _) = FD_solver_3D_elastic_quantum(Nx, Ny, Nz, dx, dy, dz, rho_model, S_matrix)
H_pauli = SparsePauliOp.from_operator(Operator(H.toarray())) # Expensive conversion
print("Hamiltonian shape: ", H.shape)
print("Hamiltonian NNZ-Ratio: ", H.nnz/H.shape[0]**2)
print("Pauli Terms (inefficient representation): ", len(H_pauli))

# Number of grid points
N = Nx*Ny*Nz
print("Number of grid points: ", N)

# Number of qubits
n = (H.shape[0]-1).bit_length()
print("Number of qubits (for wave field): ", n)

/Users/earth_bs/Documents/3D-Elastic-Wave-Equation-Hamiltonian-Simulation/_utility.py:51: FutureWarning: Input has data type int64, but the output has been cast to float64.  In the future, the output data type will match the input. To avoid this warning, set the `dtype` parameter to `None` to have the output dtype match the input, or set it to the desired output data type.
  return (1 / dx) * sp.diags([-1, 1], [0, -1], shape=(Nx, Nx), format='lil')[:, :-1]


In [ ]:
# -- Initial Conditions (3D Elastic) --
# State vector: [v_x, v_y, v_z, σ_xx, σ_yy, σ_zz, σ_xy, σ_xz, σ_yz]

f0 = 2.0            # Central frequency of the Ricker wavelet
x0, y0, z0 = 0.0, 0.0, 0.0   # Wavelet center

# Create 3D meshgrid for v_x grid (Nx-1, Ny, Nz)
# v_x is on staggered grid in x-direction
x_vx = np.linspace(-1 + dx/2, 1 - dx/2, Nx-1)
y_vx = np.linspace(-1, 1, Ny)
z_vx = np.linspace(-1, 1, Nz)
X_vx, Y_vx, Z_vx = np.meshgrid(x_vx, y_vx, z_vx, indexing='ij')

# Initialize velocity components (all zeros initially)
v0x = np.zeros((Nx-1, Ny, Nz))
v0y = np.zeros((Nx, Ny-1, Nz))
v0z = np.zeros((Nx, Ny, Nz-1))

# Initialize stress components (all zeros initially)
sigma_xx = np.zeros((Nx, Ny, Nz))
sigma_yy = np.zeros((Nx, Ny, Nz))
sigma_zz = np.zeros((Nx, Ny, Nz))
sigma_xy = np.zeros((Nx-1, Ny-1, Nz))
sigma_xz = np.zeros((Nx-1, Ny, Nz-1))
sigma_yz = np.zeros((Nx, Ny-1, Nz-1))

# Add a Ricker wavelet source to v_x component similarly to the acoustic case
# Ricker function takes (f, x, y, z, x0, y0, z0) and returns a scalar or array
ricker_vx = Ricker(f0, X_vx, Y_vx, Z_vx, x0, y0, z0)
v0x = np.round(ricker_vx, 20)

# Stack the initial conditions in the correct order:
# [v_x, v_y, v_z, σ_xx, σ_yy, σ_zz, σ_xy, σ_xz, σ_yz]
phi_0 = np.concatenate([
    v0x.flatten(),      # v_x: (Nx-1)*Ny*Nz
    v0y.flatten(),      # v_y: Nx*(Ny-1)*Nz
    v0z.flatten(),      # v_z: Nx*Ny*(Nz-1)
    sigma_xx.flatten(), # σ_xx: Nx*Ny*Nz
    sigma_yy.flatten(), # σ_yy: Nx*Ny*Nz
    sigma_zz.flatten(), # σ_zz: Nx*Ny*Nz
    sigma_xy.flatten(), # σ_xy: (Nx-1)*(Ny-1)*Nz
    sigma_xz.flatten(), # σ_xz: (Nx-1)*Ny*(Nz-1)
    sigma_yz.flatten()  # σ_yz: Nx*(Ny-1)*(Nz-1)
])
print(phi_0)
# Pad the initial conditions with zeros to match Hamiltonian size (power of 2)
phi_0 = np.concatenate([phi_0, np.zeros(H.shape[0] - psi_len)])

# Normalize the initial state and transform it to the energy basis
psi_0 = B_sqrt @ phi_0
norm = np.linalg.norm(psi_0)
psi_0 /= norm

# Number of non-zero initial state values
psi_0_nnz = np.sum(psi_0 != 0)
print('Initial state NNZ-Ratio:', psi_0_nnz/psi_len)

[-0.00000000e+00 -0.00000000e+00 -0.00000000e+00 -0.00000000e+00
 -0.00000000e+00 -2.14177310e-12 -2.14177310e-12 -0.00000000e+00
 -0.00000000e+00 -2.14177310e-12 -2.14177310e-12 -0.00000000e+00
 -0.00000000e+00 -0.00000000e+00 -0.00000000e+00 -0.00000000e+00
 -0.00000000e+00 -7.72000000e-18 -7.72000000e-18 -0.00000000e+00
 -7.72000000e-18 -2.56232681e-03 -2.56232681e-03 -7.72000000e-18
 -7.72000000e-18 -2.56232681e-03 -2.56232681e-03 -7.72000000e-18
 -0.00000000e+00 -7.72000000e-18 -7.72000000e-18 -0.00000000e+00
 -0.00000000e+00 -0.00000000e+00 -0.00000000e+00 -0.00000000e+00
 -0.00000000e+00 -2.14177310e-12 -2.14177310e-12 -0.00000000e+00
 -0.00000000e+00 -2.14177310e-12 -2.14177310e-12 -0.00000000e+00
 -0.00000000e+00 -0.00000000e+00 -0.00000000e+00 -0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000

In [ ]:
mask_dict

{'kinetic': array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
 

In [ ]:
def fracture_energy_plot(end_time, num_t=1, layers=8):
    # List of all times for evolutions
    t_array = np.linspace(0, end_time, num_t+1)
    t_array = np.delete(t_array, 0) # t = 0 will give an error when calling solve_ivp

    # Intialize energy storage
    layer_dict = {}

    z_min_list = np.linspace(0, Nz, num=layers+1, dtype=int)

    # Loop over bottom, middle, and top fracture masks, respectively
    for i in range(layers):
        layer_dict[i] = {
            'quantum' : {
                'kinetic' : [],
                'potential' : [],
                'total' : []
            },
            'classical' : {
                'kinetic' : [],
                'potential' : [],
                'total' : []
            }
        }
        print(f"Calculating energies for layer k={i}...")
        # Get the subspace projector for the current region
        mask = subspaceProjector(psi_len,get_mask_indices(get_i_location(Nx,Ny,Nz,z_min_list[i],z_min_list[i+1])))
        mask_dict = threeProjectors(mask, Nx, Ny, Nz)
        # Loop over kinetic, potential, and total energy subspace projectors, respectively
        for j in range(len(mask_dict)):
            print(f"Calculating energies for {list(mask_dict.keys())[j]} energy subspace...")
            mask = mask_dict[list(mask_dict.keys())[j]]
            # Pad the mask with zeros
            mask = np.concatenate([mask, np.zeros(H.shape[0]-psi_len)])
            # Loop over all times and calculate the quantum and classical energies
            for t in t_array:
                print(f"Calculating energy for time t={t}...")
                # -------- Quantum Circuit --------
                synthesis = MatrixExponential()
                evo = PauliEvolutionGate(H_pauli, time=t, synthesis=synthesis, label='U=exp(-iH_real*t)')

                # Define the quantum registers
                reg_wave = QuantumRegister(n, 'wave')
                reg_P = QuantumRegister(1, 'P')

                # Initialize the quantum circuit
                qc = QuantumCircuit(reg_wave, reg_P)
                qc.prepare_state(psi_0, reg_wave)

                # Apply the time-evolution operator
                qc.append(evo, reg_wave)

                # Apply unitary "projection" transform (Can often be combined into fewer MCX gates)
                qc.x(reg_P)
                for d in np.nonzero(mask)[0]:
                    qc.mcx(reg_wave, target_qubit=reg_P, ctrl_state=int(d))

                # Define observable
                O1 = Pauli('I'*(n+1))
                O2 = Pauli('Z'+'I'*n)
                observable = SparsePauliOp([O1, O2], [0.5, 0.5])
                print("Observable Pauli terms: ", len(observable))

                # Simulate the quantum circuit
                estimator = StatevectorEstimator()
                job = estimator.run([(qc, observable)], precision=1e-8)
                result = job.result()
                print(f" > Expectation value: {result[0].data.evs}")
                print(f" > Metadata: {result[0].metadata}")

                # Compute the energy loss (post-processing)
                E = result[0].data.evs
                EN_QC = (1/2) * E * norm**2 * (dx * dy * dz)
                layer_dict[i]['quantum'][list(mask_dict.keys())[j]].append(EN_QC)
                
                # classical energy calculation
                phi_0_normalized = phi_0 / norm  # Normalize using the same norm as quantum
                phi_t = solve_ivp(lambda t, phi: (B_inv @ A @ phi), (0, t), phi_0_normalized, t_eval=(0,t), method='DOP853').y.T[-1] 

                # Transform to energy basis (same as quantum case)
                psi_t_classical = B_sqrt @ phi_t

                # Calculate subspace energy using the same normalization as quantum
                # The mask selects which states to measure, then we compute the probability
                masked_state = np.diag(mask) @ psi_t_classical
                # Probability of being in the masked subspace
                prob_in_subspace = np.linalg.norm(masked_state)**2 / np.linalg.norm(psi_t_classical)**2
                EN_CL = (1/2) * prob_in_subspace * norm**2 * (dx * dy * dz)
                layer_dict[i]['classical'][list(mask_dict.keys())[j]].append(EN_CL)
    
    # -------- Plots --------

    # 1. A snapshot of energies vs time for each layer {layers} rows x 3 columns (can be {layers} plots of 1 row x 3 columns or anything well-organized)
    # Each subplot contains kinetic, potential, and total energies
    # First column is energies computed from HS
    # Second column is energies computed from classical evolution
    # Third column is a ratio plot for each energy type (kinetic, potential, total), comparing quantum to classical.
    
    
    
    # 2. A snapshot of energies vs layer (or layer vs energies) for all or certain choices of time t in t_array
    # Option I: Two separated plots or a plot with two subplots for quanntum and classical calculation.
    # Each layer has three bars for kinetic, potential, and total energy.
    # Option II: Only one plot where each layer has six bars.
    
    
    # 3. Similar to 2 but interactive plots with a time t slider.


    # 4. Plots show total energy conservation (all layers) for both quantum and classical calculations.

    return layer_dict


In [ ]:
result = fracture_energy_plot(end_time=5.54*10**(-4), num_t=100, layers=8)
print(result)

Calculating energies for layer k=0...


AssertionError: 